In [3]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
import psycopg2
from math import radians, sin, cos, sqrt, atan2
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score
import mlflow
import mlflow.sklearn

warnings.filterwarnings("ignore")



def log_to_postgres(model_name, recall, precision, f1_score, roc_auc, run_id=None):
    try:
        import psycopg2
        import numpy as np
        
        
        recall = float(recall) if isinstance(recall, np.floating) else recall
        precision = float(precision) if isinstance(precision, np.floating) else precision
        f1_score = float(f1_score) if isinstance(f1_score, np.floating) else f1_score
        roc_auc = float(roc_auc) if isinstance(roc_auc, np.floating) else roc_auc
        
        conn = psycopg2.connect(
            host="localhost",
            port=5432,
            database="fraud_db",
            user="fraud_user",
            password="fraud_pass"
        )
        cur = conn.cursor()
        
        
        cur.execute("""
            CREATE TABLE IF NOT EXISTS model_metrics (
                id SERIAL PRIMARY KEY,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                model_name VARCHAR(100),
                recall FLOAT,
                precision FLOAT,
                f1_score FLOAT,
                roc_auc FLOAT
            )
        """)
        
        
        cur.execute("""
            INSERT INTO model_metrics (model_name, recall, precision, f1_score, roc_auc)
            VALUES (%s, %s, %s, %s, %s)
        """, (model_name, recall, precision, f1_score, roc_auc))
        
        conn.commit()
        print(f"Метрики для {model_name} сохранены в PostgreSQL")
        print(f"   Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1_score:.4f}, ROC-AUC: {roc_auc:.4f}")
        
        cur.close()
        conn.close()
        return True
    except Exception as e:
        print(f"PostgreSQL ошибка: {e}")
        return False


mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("fraud_detection")


train_df = pd.read_csv("fraudTrain.csv")
test_df = pd.read_csv("fraudTest.csv")
df = pd.concat([train_df, test_df], ignore_index=True)
print(f"Загружено данных: {df.shape}")


cols_to_drop = [
    "Unnamed: 0", "cc_num", "first", "last", "street", "city",
    "state", "job", "dob", "trans_num", "merchant"
]
df = df.drop(columns=cols_to_drop, errors="ignore")

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

df["distance_km"] = df.apply(
    lambda row: haversine(row["lat"], row["long"], row["merch_lat"], row["merch_long"]),
    axis=1
)

df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["hour"] = df["trans_date_trans_time"].dt.hour


features = [
    "amt", "city_pop", "lat", "long", "merch_lat", "merch_long",
    "hour", "distance_km", "category", "gender"
]
target = "is_fraud"

X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Мошенничества в test: {int(y_test.sum())}")


enc_category = LabelEncoder()
enc_gender = LabelEncoder()

X_train["category_encoded"] = enc_category.fit_transform(X_train["category"])
X_test["category_encoded"] = enc_category.transform(X_test["category"])

X_train["gender_encoded"] = enc_gender.fit_transform(X_train["gender"])
X_test["gender_encoded"] = enc_gender.transform(X_test["gender"])

X_train = X_train.drop(columns=["category", "gender"])
X_test = X_test.drop(columns=["category", "gender"])


numeric_cols = [
    "amt", "city_pop", "lat", "long", "merch_lat",
    "merch_long", "hour", "distance_km"
]

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])


model = joblib.load("ensemble_my_models.pkl")

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "recall": recall_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba)
}

print("Результаты модели:")
for k, v in metrics.items():
    print(f"   {k}: {v:.4f}")


with mlflow.start_run(run_name="Ensemble_Model_Final") as run:
    mlflow.log_param("model_type", "VotingClassifier_soft")
    mlflow.log_param("n_models", 5)
    mlflow.log_param("weights", "[3,2,2,1,1]")
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("fraud_count_test", int(y_test.sum()))
    
    for name, value in metrics.items():
        mlflow.log_metric(name, float(value))

    mlflow.sklearn.log_model(model, name="ensemble_model")

    run_id = run.info.run_id
    print(f"Run ID: {run_id}")
    print(f"MLflow UI: http://localhost:5001")



log_to_postgres(
    model_name="Ensemble_5_models",
    recall=metrics['recall'],
    precision=metrics['precision'],
    f1_score=metrics['f1'],
    roc_auc=metrics['roc_auc'],
    run_id=run_id
)


results = {
    "model": "Ensemble_5_models",
    "metrics": metrics,
    "run_id": run_id,
    "mlflow_ui": "http://localhost:5001",
    "timestamp": str(pd.Timestamp.now())
}

with open("experiment_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("Результаты сохранены в 'experiment_results.json'")


try:
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        database="fraud_db",
        user="fraud_user",
        password="fraud_pass"
    )
    cur = conn.cursor()
    cur.execute("""
        SELECT timestamp, model_name, recall, precision, f1_score, roc_auc
        FROM model_metrics 
        ORDER BY timestamp DESC 
        LIMIT 5
    """)
    
    
    for row in cur.fetchall():
        print(f"  {row[0]} | {row[1]} | R:{row[2]:.4f} | P:{row[3]:.4f} | F1:{row[4]:.4f}")
    
    cur.close()
    conn.close()
except Exception as e:
    print(f" Не удалось проверить PostgreSQL: {e}")


print(f"""
   Лучшая модель: Ensemble (Voting Classifier)

   Метрики на тестовой выборке:
   Recall:     {metrics['recall']:.4f} ({metrics['recall']*100:.2f}%)
   Precision:  {metrics['precision']:.4f} ({metrics['precision']*100:.2f}%)
   F1-score:   {metrics['f1']:.4f}
   ROC-AUC:    {metrics['roc_auc']:.4f}

   Ссылки:
   MLflow UI:    http://localhost:5001
   PostgreSQL:   localhost:5432 (fraud_db/fraud_user)
   Grafana:      http://localhost:3000

   Сохраненные файлы:
   experiment_results.json
   ensemble_my_models.pkl
""")


2026/06/02 08:54:45 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection' does not exist. Creating a new experiment.


Загружено данных: (1852394, 23)
Train: (1481915, 10), Test: (370479, 10)
Мошенничества в test: 1930
Результаты модели:
   recall: 0.9140
   precision: 0.1232
   f1: 0.2171
   roc_auc: 0.9893


2026/06/02 08:58:27 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/06/02 08:58:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle form

Run ID: 2d5253586d164208a23328e9cbad5050
MLflow UI: http://localhost:5001
🏃 View run Ensemble_Model_Final at: http://localhost:5001/#/experiments/1/runs/2d5253586d164208a23328e9cbad5050
🧪 View experiment at: http://localhost:5001/#/experiments/1
Метрики для Ensemble_5_models сохранены в PostgreSQL
   Recall: 0.9140, Precision: 0.1232, F1: 0.2171, ROC-AUC: 0.9893
Результаты сохранены в 'experiment_results.json'
  2026-06-02 05:59:57.028568 | Ensemble_5_models | R:0.9140 | P:0.1232 | F1:0.2171

   Лучшая модель: Ensemble (Voting Classifier)

   Метрики на тестовой выборке:
   Recall:     0.9140 (91.40%)
   Precision:  0.1232 (12.32%)
   F1-score:   0.2171
   ROC-AUC:    0.9893

   Ссылки:
   MLflow UI:    http://localhost:5001
   PostgreSQL:   localhost:5432 (fraud_db/fraud_user)
   Grafana:      http://localhost:3000

   Сохраненные файлы:
   experiment_results.json
   ensemble_my_models.pkl



In [2]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="fraud_db",
    user="fraud_user",
    password="fraud_pass"
)

cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS prediction_logs (
    id SERIAL PRIMARY KEY,
    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    user_id VARCHAR(100),
    amount FLOAT,
    city_pop FLOAT,
    lat FLOAT,
    lon FLOAT,
    merch_lat FLOAT,
    merch_long FLOAT,
    hour INT,
    distance_km FLOAT,
    is_fraud BOOLEAN,
    probability FLOAT,
    risk_message TEXT
)
""")

conn.commit()

cur.close()
conn.close()